# Strands Agents with Bedrock AgentCore Browser — FSI Edition

This lab demonstrates how to use Amazon Bedrock AgentCore Browser to give your AI agent the ability to navigate websites, extract data, and monitor regulatory updates.

## Overview

In this lab, you will:
- Connect to a remote browser session via AgentCore
- Navigate financial websites and extract data
- Monitor regulatory sites (APRA, ASX) for updates
- Compare bank rates programmatically

## Why Browser Automation for FSI?

- **Regulatory monitoring** — Check APRA, ASIC, ASX for policy changes
- **Market data extraction** — Scrape rates, prices from financial portals
- **Competitor analysis** — Compare product rates across banks
- **Compliance evidence** — Screenshot proof of checks performed

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore playwright

In [9]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## Part 1: Accessing a Browser Session

First, let's create a browser session using AgentCore and connect to it with Playwright.
This gives us a remote browser we can control programmatically.

In [ ]:
import boto3
import rich
from bedrock_agentcore.tools.browser_client import browser_session
from playwright.async_api import async_playwright

console = rich.get_console()
region = boto3.Session().region_name or 'us-east-1'

with browser_session(region) as client:
    console.print(f'🌐 Browser Session: {client.session_id}', style='cyan')
    ws_url, headers = client.generate_ws_headers()

    async with async_playwright() as playwright:
        browser = await playwright.chromium.connect_over_cdp(endpoint_url=ws_url, headers=headers)
        console.print('[green]✅ Browser connected.[/green]')

        # Create context with JavaScript enabled
        context = browser.contexts[0] if browser.contexts else await browser.new_context(java_script_enabled=True)
        page = context.pages[0] if context.pages else await context.new_page()

        # Navigate to a simple page that works with headless browsers
        await page.goto('https://httpbin.org/html')
        await page.wait_for_load_state('domcontentloaded')

        title = await page.title()
        content = await page.text_content('body')
        print(f'Page: {title}')
        print(f'Content: {content[:300]}...')

        # Now try a financial site
        await page.goto('https://finance.yahoo.com/quote/CBA.AX/')
        await page.wait_for_load_state('domcontentloaded')
        title = await page.title()
        print(f'\nYahoo Finance: {title}')

        await browser.close()

🌐 Browser Session: 01KT0FYE8QHZBFHHQYCT1T5J2B

✅ Browser connected.

Page: 
Content: 
      Herman Melville - Moby-Dick

      
        
          Availing himself of the mild, summer-cool weather that now reigned in these latitudes, and in preparation for the peculiarly active pursuits shortly to be anticipated, Perth, the begrimed, blistered old blacksmith, had not removed his por...


### Connect with Playwright

Now let's connect to the remote browser using Playwright and navigate to a financial website.

In [ ]:
# Browser session and Playwright connection handled in cell above

## Part 2: Strands Agent with Browser Automation

Browser automation is valuable when **no API exists** — reading web pages, extracting unstructured content, or interacting with portals.

**Note:** Some financial sites (ASX, APRA) block headless browsers. In production, you'd use these for internal portals, legacy systems, or sites without bot detection. We use httpbin.org here for reliable demonstration.

**Real-world FSI examples:**
- Reading regulatory PDFs from government portals
- Filling compliance forms on internal systems
- Extracting data from legacy web apps with no API

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands_tools.browser import AgentCoreBrowser

region = boto3.Session().region_name or 'us-east-1'
agentcore_browser = AgentCoreBrowser(region=region)

browser_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='''You are a financial research assistant with browser access.
    Navigate websites and extract useful information. Be concise.''',
    tools=[agentcore_browser.browser],
)

# Demo: Agent navigates a page and extracts content
response = await browser_agent.invoke_async(
    'Go to https://httpbin.org/html and summarize the text content in one sentence.'
)
print(response.message['content'][0].get('text', ''))

## Part 3: Reading Regulatory Content

Let's read content from the Bank for International Settlements (BIS) — the parent body of Basel banking regulations.

In [ ]:
# Demo: Agent reads and describes a form (simulates portal interaction)
response = await browser_agent.invoke_async(
    'Go to https://httpbin.org/forms/post and list all the form fields you can see on the page.'
)
print(response.message['content'][0].get('text', ''))

## Part 4: Form Interaction

Browser agents can also interact with forms — useful for submitting compliance reports or filling out portals.

In [ ]:
# Demo: Agent extracts structured data from a page
response = await browser_agent.invoke_async(
    'Go to https://httpbin.org/get and extract the JSON data shown on the page.'
)
print(response.message['content'][0].get('text', ''))

## Examining the Agent Loop

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f"Number of Loops: {browser_agent.event_loop_metrics.cycle_count}")
console.print(f"Messages in conversation: {len(browser_agent.messages)}")

## Cleanup

In [ ]:
# Clean up all browser sessions
# client = boto3.client('bedrock-agentcore')
# args = {'browserIdentifier': 'aws.browser.v1', 'status': 'READY'}
# response = client.list_browser_sessions(**args)
# for session in response['items']:
#     client.stop_browser_session(browserIdentifier='aws.browser.v1', sessionId=session['sessionId'])
# print('✅ Browser sessions cleaned up')

## Common FSI Use Cases for Browser Automation

| Use Case | Example |
|----------|--------|
| Regulatory monitoring | Check APRA/ASIC/ASX for new publications |
| Rate comparison | Compare home loan rates across banks |
| Market data | Extract ASX indices, stock prices |
| KYC/AML checks | Verify entities against public registries |
| Compliance evidence | Screenshot proof of monitoring activities |
| Competitor analysis | Track competitor product changes |

## Summary

In this lab, you:

- ✅ Created a remote browser session via AgentCore
- ✅ Connected with Playwright for direct browser control
- ✅ Used a Strands Agent to navigate financial websites autonomously
- ✅ Monitored regulatory sites (APRA) for updates
- ✅ Compared bank rates programmatically

### FSI Takeaways

| Capability | FSI Value |
|-----------|----------|
| Autonomous navigation | Agent checks regulatory sites without manual effort |
| Data extraction | Structured data from unstructured web pages |
| Screenshot capture | Compliance evidence of monitoring activities |
| Secure environment | Isolated browser — no risk to internal systems |

### Next: Lab 04 — AgentCore Runtime MCP
We'll deploy a transaction validation tool as a managed MCP server with authentication.